# h5py 教程：高效存储科学计算数据

h5py 是 Python 中操作 HDF5 格式文件的核心库。HDF5（Hierarchical Data Format）是一种专为大规模科学数据设计的存储格式，特别适合存储多维数组和结构化数据。

> **核心记忆点**：`Group` 行为像字典，`Dataset` 行为像 NumPy 数组

In [ ]:
# 安装依赖（如果尚未安装）
# !pip install h5py numpy

In [ ]:
import h5py
import numpy as np
import os

print(f"h5py version: {h5py.__version__}")
print(f"NumPy version: {np.__version__}")

---
## 1. 基本概念：HDF5 文件结构

HDF5 文件包含两类对象：
- **Group（组）**：类似文件夹，用于组织数据
- **Dataset（数据集）**：类似多维数组，实际存储数据
- **Attribute（属性）**：附加在 Group 或 Dataset 上的元数据

---
## 2. 创建和打开文件

In [ ]:
# 文件访问模式
# 'r'  - 只读（默认）
# 'r+' - 读写，文件必须已存在
# 'w'  - 写入，若文件存在则清空重写
# 'w-' / 'x' - 写入，文件必须不存在
# 'a'  - 追加（默认打开，不存在则创建）

# 使用上下文管理器创建文件
with h5py.File('tutorial.hdf5', 'w') as f:
    print("文件创建成功！")

In [ ]:
# 检查文件是否存在
filepath = 'tutorial.hdf5'
print(f"文件存在: {os.path.exists(filepath)}")
print(f"文件大小: {os.path.getsize(filepath)} bytes")

---
## 3. 创建 Dataset

In [ ]:
with h5py.File('tutorial.hdf5', 'w') as f:
    # 方法1：指定 shape 和 dtype，不填充数据
    dset1 = f.create_dataset('dataset1', (100,), dtype='i')
    
    # 方法2：直接传入 NumPy 数组（自动推断 shape 和 dtype）
    data = np.arange(100)
    dset2 = f.create_dataset('dataset2', data=data)
    
    # 方法3：创建多维数组
    matrix = np.random.rand(10, 20)
    f.create_dataset('matrix', data=matrix)
    
    # 方法4：用赋值语法创建（等价于 create_dataset(data=...)
    f['simple'] = np.array([1, 2, 3, 4, 5])

print("Datasets 创建完成！")

### Dataset 的基本属性

In [ ]:
with h5py.File('tutorial.hdf5', 'r') as f:
    dset = f['matrix']
    print(f"名称: {dset.name}")
    print(f"形状: {dset.shape}")
    print(f"数据类型: {dset.dtype}")
    print(f"元素个数: {dset.size}")
    print(f"维度数: {dset.ndim}")
    print(f"占用字节数: {dset.nbytes}")

---
## 4. 读取和写入数据

In [ ]:
with h5py.File('tutorial.hdf5', 'r+') as f:
    # 读取整个 dataset → 返回 NumPy 数组
    data = f['dataset1'][:]
    print(f"读取 dataset1 全部数据: {data[:10]}...")
    
    # 读取部分数据（切片）
    data_slice = f['dataset2'][10:20]
    print(f"读取 dataset2[10:20]: {data_slice}")
    
    # 标量读取
    first_elem = f['dataset2'][0]
    print(f"第一个元素: {first_elem}")
    
    # 读取标量 dataset（用空元组 ()）
    scalar = np.float64(3.14)
    f.create_dataset('scalar', data=scalar)
    val = f['scalar'][()]
    print(f"标量值: {val}")

In [ ]:
with h5py.File('tutorial.hdf5', 'r+') as f:
    # 写入数据（切片赋值）
    f['dataset1'][:] = np.arange(100, 200)
    print("写入 dataset1 完成")
    
    # 分片写入
    f['dataset1'][0:10] = np.arange(10) * 10
    print("写入 dataset1[0:10] 完成")
    
    # 多维切片写入
    f['matrix'][0:5, 0:5] = np.ones((5, 5))
    print("写入 matrix[0:5, 0:5] 完成")
    
    # 广播赋值
    f['matrix'][:, 0] = np.arange(10)
    print("广播赋值完成")

In [ ]:
# ⚠️ 常见错误：两次索引赋值
# dset[0][1] = 3.0  ← 只修改了内存中的临时数组，不会写入文件！
# 正确写法：
with h5py.File('tutorial.hdf5', 'r+') as f:
    dset = f.create_dataset('test', (3, 3), dtype='f')
    dset[0, 1] = 3.0
    print(f"正确赋值: {dset[0, 1]}")

---
## 5. Group（组）的操作

In [ ]:
# Group 类似文件夹，支持层次化存储
with h5py.File('tutorial.hdf5', 'w') as f:
    # 创建子组
    grp1 = f.create_group('experiment1')
    grp2 = f.create_group('experiment2')
    
    # 在子组中创建 dataset
    grp1.create_dataset('data', data=np.random.rand(10, 10))
    grp2.create_dataset('data', data=np.random.rand(5, 5))
    
    # 嵌套多层组
    sub_grp = grp1.create_group('subgroup')
    sub_grp.create_dataset('nested', data=np.array([1, 2, 3]))
    
    # 直接通过路径创建（中间组会自动创建）
    f.create_dataset('auto/path/created', data=np.array([4, 5, 6]))

print("Group 结构创建完成！")

In [ ]:
with h5py.File('tutorial.hdf5', 'r') as f:
    # 查看顶级 keys
    print(f"顶级 keys: {list(f.keys())}")
    
    # 通过路径访问
    dset = f['experiment1/data']
    print(f"实验1数据形状: {dset.shape}")
    
    # 遍历所有条目
    print("\n使用 visit 遍历：")
    def print_name(name):
        print(f"  {name}")
    f.visit(print_name)
    
    # 检查是否存在某个路径
    print(f"\n' experiment1/data' 存在: {'experiment1/data' in f}")

---
## 6. Attribute（属性）

In [ ]:
# Attribute 是附加在 Group/Dataset 上的小型元数据字典
with h5py.File('tutorial.hdf5', 'w') as f:
    dset = f.create_dataset('measurements', data=np.random.rand(5, 5))
    
    # 添加标量属性
    dset.attrs['unit'] = 'Kelvin'
    dset.attrs['temperature'] = 300.0
    dset.attrs['experiment_id'] = 42
    
    # 添加数组属性
    dset.attrs['calibration'] = np.array([1.0, 2.0, 3.0])
    
    # 添加字符串属性（需要指定 dtype）
    dset.attrs['description'] = np.array(['Neural Network QMC'], dtype=h5py.string_dtype())
    
    # 在组上添加属性
    grp = f.create_group('config')
    grp.attrs['version'] = '1.0'
    grp.attrs['author'] = 'your_name'

In [ ]:
with h5py.File('tutorial.hdf5', 'r') as f:
    dset = f['measurements']
    
    # 读取属性
    print(f"单位: {dset.attrs['unit']}")
    print(f"温度: {dset.attrs['temperature']}")
    print(f"实验编号: {dset.attrs['experiment_id']}")
    print(f"校准值: {dset.attrs['calibration']}")
    print(f"描述: {dset.attrs['description']}")
    
    # 查看所有属性键
    print(f"\n所有属性键: {list(dset.attrs.keys())}")
    
    # 安全读取（带默认值）
    val = dset.attrs.get('nonexistent', 'default_value')
    print(f"不存在的属性: {val}")

---
## 7. 压缩与分块存储

In [ ]:
# 大文件存储时，压缩和分块可以显著提高效率
with h5py.File('tutorial.hdf5', 'w') as f:
    # 使用 gzip 压缩（压缩比高，速度适中）
    dset_compressed = f.create_dataset(
        'compressed_data',
        data=np.random.rand(100, 100),
        compression='gzip',
        compression_opts=4  # 压缩级别 1-9
    )
    
    # 使用 lzf 压缩（压缩率低，但速度最快）
    dset_lzf = f.create_dataset(
        'lzf_data',
        data=np.random.rand(100, 100),
        compression='lzf'
    )
    
    # 分块存储（适合频繁访问部分数据）
    dset_chunked = f.create_dataset(
        'chunked_data',
        shape=(1000, 1000),
        dtype='f',
        chunks=(100, 100),  # 每块 100x100
        compression='gzip',
        compression_opts=2
    )

print(f"文件大小: {os.path.getsize('tutorial.hdf5')} bytes")

In [ ]:
# 查看压缩和分块信息
with h5py.File('tutorial.hdf5', 'r') as f:
    for name in ['compressed_data', 'lzf_data', 'chunked_data']:
        dset = f[name]
        print(f"\n{name}:")
        print(f"  压缩: {dset.compression}")
        print(f"  分块: {dset.chunks}")

---
## 8. 实际示例：存储神经网络量子态训练数据

In [ ]:
# 模拟存储 VMC 训练过程中的数据
with h5py.File('vmc_training.hdf5', 'w') as f:
    # 存储超参数
    f.attrs['model'] = 'NES_VMC'
    f.attrs['system'] = 'LiH'
    f.attrs['created_at'] = '2026-08-30'
    
    # 存储训练曲线（能量随步数变化）
    n_steps = 10000
    n_energies = 100
    energies = np.cumsum(np.random.randn(n_steps) * 0.01) + np.linspace(0, -0.5, n_steps)
    
    # 按 epoch 存储
    grp = f.create_group('training')
    grp.create_dataset('energies', data=energies)
    grp.create_dataset('epoch', data=np.linspace(0, 100, n_steps))
    
    # 存储每个参数的矩阵（权重）
    n_params = 500
    grp.create_dataset('weights', data=np.random.randn(n_params, 100))
    
    # 存储采样配置
    cfg = grp.create_group('config')
    cfg.attrs['n_chains'] = 64
    cfg.attrs['sweep_size'] = 100
    cfg.create_dataset('energies_per_step', data=np.random.randn(n_steps, n_chains))

print("VMC 训练数据创建完成！")
print(f"文件大小: {os.path.getsize('vmc_training.hdf5')} bytes")

In [ ]:
# 读取存储的 VMC 数据
with h5py.File('vmc_training.hdf5', 'r') as f:
    print(f"模型: {f.attrs['model']}")
    print(f"体系: {f.attrs['system']}")
    
    grp = f['training']
    print(f"\n能量数据形状: {grp['energies'].shape}")
    print(f"权重数据形状: {grp['weights'].shape}")
    print(f"每个 step 的能量形状: {grp['config']['energies_per_step'].shape}")
    print(f"链条数: {grp['config'].attrs['n_chains']}")

---
## 9. 常见注意事项

In [ ]:
# 1. Dataset 不可变性
# shape 和 dtype 一旦创建不能修改
# 但可以 resize（如果允许扩展）
with h5py.File('tutorial.hdf5', 'r+') as f:
    dset = f.create_dataset('resizable', (10,), dtype='f', maxshape=(None,))
    dset.resize((20,))
    print(f"resize 后形状: {dset.shape}")
    
# 2. 字符串编码
# 默认 ASCII，中文需要 string_dtype
with h5py.File('tutorial.hdf5', 'r+') as f:
    f.attrs['中文键'] = np.array(['你好'], dtype=h5py.string_dtype())
    print(f"中文属性: {f.attrs['中文键']}")

---
## 10. 总结速查表

In [ ]:
summary = """
╔══════════════════════════════════════════════════════════╗
║              h5py 快速参考卡                              ║
╠══════════════════════════════════════════════════════════╣
║ 创建文件：h5py.File('file.hdf5', 'w')                    ║
║ 打开文件：h5py.File('file.hdf5', 'r')                    ║
║ 创建组：f.create_group('name')                           ║
║ 创建数据集：f.create_dataset('name', data=arr)          ║
║ 读取数据：f['name'][:]                                   ║
║ 写入数据：f['name'][:] = new_arr                         ║
║ 访问属性：f['name'].attrs['key']                         ║
║ 压缩存储：compression='gzip'                             ║
║ 分块存储：chunks=(100, 100)                              ║
║ 查看keys：list(f.keys())                                 ║
║ 遍历路径：f.visit(callback)                              ║
╚══════════════════════════════════════════════════════════╝
"""
print(summary)

In [ ]:
# 清理生成的文件
for fname in ['tutorial.hdf5', 'vmc_training.hdf5']:
    if os.path.exists(fname):
        os.remove(fname)
        print(f"已删除: {fname}")